# Training Notebook
This notebook for training model with 6 features and with out file stats channel
## Setup library and enviroments

In [ ]:
import os
import sys

# Thêm đường dẫn tới thư mục gốc (backend) để import app/...
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import time
import random
import os
import json
import psutil
import shutil
from IPython.core.magics.execution import ExecutionMagics

import numpy as np
import pandas as pd

from pyspark.sql import DataFrame
from pyspark.ml.feature import StringIndexer

from streaming import MDSWriter, StreamingDataset, StreamingDataLoader
from streaming.base.util import clean_stale_shared_memory
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, IterableDataset

import logging

logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("mlflow.utils.autologging_utils").setLevel(logging.ERROR)
logging.getLogger("mlflow.tracking").setLevel(logging.ERROR)
logging.getLogger("mlflow.store").setLevel(logging.ERROR)
logging.getLogger("mlflow.projects").setLevel(logging.ERROR)
logging.getLogger("mlflow.pyfunc").setLevel(logging.ERROR)
logging.getLogger("mlflow.pytorch").setLevel(logging.ERROR)

import mlflow
import mlflow.pytorch
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plt

mlflow.pytorch.autolog(disable=True)
mlflow.autolog(disable=True, log_models=False)

In [ ]:
# Sửa lại các imports từ codebase cũ sang codebase mới (app.anomaly_detection_v6)
from app.anomaly_detection_v6.models import SupportedAnomalyDetectionModels, create_model
from app.anomaly_detection_v6.criterions import SupportedAnomalyDetectionLosses, create_criterion
import app.anomaly_detection_v6.constants as constants
from app.anomaly_detection_v6.utils import set_reproducibility
from app.anomaly_detection_v6.training.datasets import RobotMosaicDataset, write_sdf_to_mds
from app.anomaly_detection_v6.training.trainer import RobotAnomalyDetectionTrainer

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Data config
WINDOW_DATA_TABLE = 'dbxws_multimodalai001.silver_layer.slicingwindow_data_training_variable_robot_indexed'
VAL_TABLE = 'dbxws_multimodalai001.silver_layer.slicingwindow_data_training_variable_robot_indexed_val'

# Training config
BATCH_SIZE = 256
NUM_WORKERS = 2
EPOCHS = 10
LR = 1e-4

W_WIN = 2.0
W_FILE = 1.5
W_VQ = 5.0
W_EDGE = 0.3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MDS config
# LƯU Ý: Đây là đường dẫn legacy trỏ đến DBFS. Chỉnh lại cho phù hợp nếu cần trỏ local cache
MDS_REMOTE_DIR = '/Volumes/dbxws_multimodalai001/bronze_layer/file/anomaly_detection_experiments/20260406-Exp/mds'
MDS_LOCAL_CACHE = './local_disk0/mds_cache'

MDS_COLUMNS = {
    'raw_window': 'ndarray:float32',
    'stats': 'ndarray:float32',
    'rel_pos': 'float32',
    'robot_idx': 'int32',
    'program_idx': 'int32'
}

MANUAL_MODEL_DIR = "/Volumes/dbxws_multimodalai001/bronze_layer/file/anomaly_detection_experiments/20260406-Exp/checkpoints"

if os.path.exists(MDS_LOCAL_CACHE):
    shutil.rmtree(MDS_LOCAL_CACHE)

In [ ]:
set_reproducibility(seed=constants.RANDOM_SEED)

## Create datasets

In [ ]:
all_columns_to_select = ['robot_code', 'program_number', 'start_time', 'window_start_serial'] + constants.STATS_COLS + ["raw_values_array"] + ["robot_idx", "program_idx"]

query = f"SELECT {', '.join(all_columns_to_select)} FROM {WINDOW_DATA_TABLE}"
sdf_indexed = spark.sql(query)

In [ ]:
sdf_indexed = sdf_indexed.filter("robot_code IN (3, 4, 5, 6, 7, 8)")
sdf_indexed.groupBy('robot_code', 'program_number', 'start_time').count().count()

In [ ]:
NUM_ROBOTS = int(sdf_indexed.select('robot_code').distinct().count()) + 1
NUM_PROGRAMS = int(sdf_indexed.select('program_number').distinct().count()) + 1

print(f"Total robot: {NUM_ROBOTS}, total program: {NUM_PROGRAMS}")

In [ ]:
file_ids = sdf_indexed.select('robot_code', 'program_number', 'start_time').distinct().orderBy('robot_code', 'program_number', 'start_time')

train_file_ids, val_file_ids = file_ids.randomSplit([0.85, 0.15], seed=constants.RANDOM_SEED)

train_sdf = sdf_indexed.join(train_file_ids, on=['robot_code', 'program_number', 'start_time'], how="inner")
val_sdf = sdf_indexed.join(val_file_ids, on=['robot_code', 'program_number', 'start_time'], how="inner")

train_sdf.cache()
val_sdf.cache()

print(f"Training files: {train_sdf.count()}")
print(f"Validation files: {val_sdf.count()}")

In [ ]:
val_sdf.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(VAL_TABLE)

write_sdf_to_mds(train_sdf, 'train', MDS_REMOTE_DIR, MDS_COLUMNS, [0, 1, 2, 3, 4, 5], enable_file_stats=False)
write_sdf_to_mds(val_sdf, 'val', MDS_REMOTE_DIR, MDS_COLUMNS, [0, 1, 2, 3, 4, 5], enable_file_stats=False)

In [ ]:
clean_stale_shared_memory()

if os.path.exists(MDS_LOCAL_CACHE):
    shutil.rmtree(MDS_LOCAL_CACHE)
os.makedirs(MDS_LOCAL_CACHE, exist_ok=True)

current_train_cache = os.path.join(MDS_LOCAL_CACHE, 'train')
if os.path.exists(current_train_cache):
    shutil.rmtree(current_train_cache)
os.makedirs(current_train_cache, exist_ok=True)

current_val_cache = os.path.join(MDS_LOCAL_CACHE, 'val')
if os.path.exists(current_val_cache):
    shutil.rmtree(current_val_cache)
os.makedirs(current_val_cache, exist_ok=True)

train_ds = RobotMosaicDataset(
    remote=os.path.join(MDS_REMOTE_DIR, 'train'),
    local=os.path.join(MDS_LOCAL_CACHE, 'train'),
    shuffle=True,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

val_ds = RobotMosaicDataset(
    remote=os.path.join(MDS_REMOTE_DIR, 'val'),
    local=os.path.join(MDS_LOCAL_CACHE, 'val'),
    shuffle=False,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

train_loader = StreamingDataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

val_loader = StreamingDataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

start = time.time()
sample_batch = next(iter(train_loader))
raw_window_batch = sample_batch['raw_window']
stats_batch = sample_batch['stats']

print(f"Batch loaded in {time.time() - start:.2f} seconds")
print(f"Shape of raw_window: {raw_window_batch.shape}")
print(f"Shape of stats: {stats_batch.shape}")

## Training

In [ ]:
# Chú ý thay đổi tên biến config khớp với tên param mới trong models (_num_sensors, _num_win_stats_features,...)
metadata_dict = dict(
    num_sensors=constants.NUM_SENSORS,
    window_size=constants.WINDOW_SIZE,
    num_win_stats_features=constants.NUM_STATS,
    num_robots=constants.MAX_ROBOTS,
    num_programs=constants.MAX_PROGRAMS,
    d_model=128, codebook_size=128, num_mixer_blocks=2, num_vq_levels=2, dropout=0.1,
)

model = create_model(SupportedAnomalyDetectionModels.V6_RVQ, metadata_dict)

criterion_metadata = dict(w_win=W_WIN, w_vq=W_VQ, w_edge=W_EDGE)
criterion = create_criterion(SupportedAnomalyDetectionLosses.MULTITASK_VQ, criterion_metadata)

print(model.get_metadata())

# Tính steps cho trainer
print("Estimating step ...")
train_steps = train_sdf.count() // BATCH_SIZE + 1
val_steps = val_sdf.count() // BATCH_SIZE + 1

# Khởi tạo và gọi Trainer thay cho đoạn code Training loop dài dòng cũ
trainer = RobotAnomalyDetectionTrainer(
    model=model,
    criterion=criterion,
    lr=LR,
    batch_size=BATCH_SIZE,
    use_mlflow=True,
    mlflow_run_name="20260404-Exp",
    manual_checkpoint_dir=MANUAL_MODEL_DIR,
    device=DEVICE
)

trainer(
    train_dl=train_loader,
    val_dl=val_loader,
    epochs=EPOCHS,
    train_steps=train_steps,
    val_steps=val_steps
)